In [16]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.training import train_state

In [3]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

In [22]:
X_train = raw_data[:40,:].T
y_train = raw_data[145,:].T - 1
y_train = jnp.array(y_train, dtype=jnp.int32)
print(f'Checking shape: {X_train.shape, y_train.shape}')

Checking shape: ((39512, 40), (39512,))


In [25]:
TIME_STEPS = 127
BATCH_SIZE = 32

dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_train,
    targets=y_train[TIME_STEPS - 1:], 
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
)

dataset = dataset.prefetch(tf.data.AUTOTUNE)

for batch_x, batch_y in dataset.take(1):
    print(f"抽取出的 X 张量形状: {batch_x.shape}")
    print(f"抽取出的 Y 张量形状: {batch_y.shape}")

抽取出的 X 张量形状: (32, 127, 40)
抽取出的 Y 张量形状: (32,)


In [11]:
class TCNBlock(nn.Module):
    features: int
    dilation: int
    kernel_size: int = 3

    @nn.compact 
    def __call__(self, x):
        y = nn.Conv(
            features=self.features,
            kernel_size=(self.kernel_size,),
            kernel_dilation=(self.dilation,),
            padding='CAUSAL'
        )(x)
        return nn.relu(y)

class TCN(nn.Module):
    features: int = 64

    @nn.compact
    def __call__(self, x):
        d = [1, 2, 4, 8, 16, 32]
        
        for dilation in d:
            x = TCNBlock(
                features=self.features, 
                dilation=dilation, 
                kernel_size=3
            )(x)
            
        Output = x[:, -1, :]
        logits = nn.Dense(features=3)(Output)
        return logits

In [15]:
def create_train_state(rng, model, learning_rate = 0.01):
    dummy_x = jnp.ones((1, 100, 40)) 
    variables = model.init(rng, dummy_x)
    params = variables['params'] 

    tx = optax.adam(learning_rate)

    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
    )

In [17]:
@jax.jit
def train_step(state, batch_x, batch_y):
    """
    这就是被 @jax.jit 封印的终极函数！
    它在第一次运行时会被编译成 C++/XLA 底层代码，后续调用速度会起飞。
    """
    
    # 1. 定义我们要最优化的目标函数 (Loss Function)
    def loss_fn(params):
        # 执行前向传播，得到 3 个类别的未归一化分数 (logits)
        logits = state.apply_fn({'params': params}, batch_x)
        
        # 计算交叉熵损失 (针对 0:涨, 1:平, 2:跌 这种互斥分类任务的最佳目标函数)
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits=logits, labels=batch_y
        ).mean() # 取这个 Batch 的平均 Loss
        
        # 顺便计算一下准确率，方便监控 (辅助输出)
        preds = jnp.argmax(logits, axis=-1)
        accuracy = jnp.mean(preds == batch_y)
        
        # JAX 规定：求导函数必须返回一个标量 (loss) 作为第一个返回值
        return loss, accuracy 
        
    # 2. 自动微分：jax.value_and_grad 是 JAX 的皇冠明珠
    # has_aux=True 表示我们的 loss_fn 除了 loss，还返回了额外的辅助信息 (accuracy)
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    
    # 求解出当前的 loss、accuracy，以及所有权重的梯度 (grads)
    (loss, accuracy), grads = grad_fn(state.params)
    
    # 3. 梯度下降：根据算出的梯度，更新模型的状态 (类似于 W_new = W_old - lr * grad)
    state = state.apply_gradients(grads=grads)
    
    # 返回更新后的全新 state，以及监控用的 loss 和 accuracy
    return state, loss, accuracy

print("train_step 静态函数编译指令已就绪！")

train_step 静态函数编译指令已就绪！


In [18]:
rng = jax.random.PRNGKey(42) # JAX 极度依赖显式的随机种子
tcn_model = TCN() # 假设这是你刚才写好的那个 TCN 类
state = create_train_state(rng, tcn_model, learning_rate=1e-3)
print("模型状态容器打包完毕！包含所有的网络权重和 Adam 动量信息。")

模型状态容器打包完毕！包含所有的网络权重和 Adam 动量信息。


In [20]:
import time

In [26]:
EPOCHS = 3

print(f"--- 🚀 引擎点火：开始 {EPOCHS} 个 Epoch 的高频 TCN 训练 ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 用于记录当前 Epoch 的累计状态
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    # 遍历我们之前做好的 1232 个 Batch 的 tf.data 管道
    for batch_x, batch_y in dataset:
        
        # ⚠️ 极其关键的“次元壁突破”：
        # dataset 吐出来的是 TensorFlow 张量，我们需要用 .numpy() 把它们变成纯粹的内存数组，
        # 然后再用 jnp.array 喂给 JAX。这一步在底层是零拷贝的，速度极快。
        x_jax = jnp.array(batch_x.numpy())
        y_jax = jnp.array(batch_y.numpy())
        
        # 核心爆炸输出：执行单步训练！
        # state 被送进去，伴随着 XLA 机器码的疯狂运算，吐出一个更新了权重的全新 state
        state, loss, accuracy = train_step(state, x_jax, y_jax)
        
        # 累加指标
        epoch_loss += loss
        epoch_accuracy += accuracy
        batch_count += 1
        
        # 打印一下第一个 Batch 的进度，证明计算图已经打通并开始求导
        if batch_count == 1 and epoch == 0:
            print(f"✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

    # 计算整个 Epoch 的平均指标
    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | 耗时: {epoch_time:.2f}s | 平均 Loss: {avg_loss:.4f} | 平均 Accuracy: {avg_acc:.4f}")

print("--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---")

--- 🚀 引擎点火：开始 3 个 Epoch 的高频 TCN 训练 ---
✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: 1.1170, Accuracy: 0.2500
Epoch 1/3 | 耗时: 30.96s | 平均 Loss: 1.0093 | 平均 Accuracy: 0.5203
Epoch 2/3 | 耗时: 30.65s | 平均 Loss: 1.0154 | 平均 Accuracy: 0.5229
Epoch 3/3 | 耗时: 29.56s | 平均 Loss: 1.0074 | 平均 Accuracy: 0.5224
--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---
